# 🗺️ Planning vs. ReAct: Plan First, Then Execute

## Learning Objectives
In this notebook, you will learn:
1. **The planning pattern** - What it means to decompose a task into a plan *before* acting, and when that pays off
2. **A ReAct baseline** - Run a reactive agent that decides its next step after every tool result
3. **A plan-and-execute agent** - Build a Planner → Executor → Synthesizer graph in LangGraph
4. **A measured comparison** - Count LLM calls, tool calls, tokens and wall time for both agents in code
5. **Judging the answers** - Have an LLM copy the evidence from each answer, compute the verdict in code, and prove the judge can fail an answer

## Where this fits
Modules 05–09 were *workflows*: you drew every step. From here on the model writes part of the control flow itself. This notebook contrasts the two ways it can do that - one step at a time (ReAct) or all steps up front (a plan). `02_planning_decompose_execute_replan.ipynb` then lets the plan change when a step fails.

## Prerequisites
- The ReAct loop in LangGraph - `modules/03_react_agents/notebooks/01_react_agent_langgraph.ipynb` (and, from scratch, `modules/01_agents_from_scratch/notebooks/03_planning_pattern_react.ipynb`)
- `with_structured_output` - `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env`

---
## 📖 Part 1: The Planning Pattern

The **Planning** architecture introduces a layer of foresight into an agent's reasoning. Instead of reacting to information step by step as in the ReAct model, a planning agent first decomposes a complex task into a sequence of smaller, manageable sub-goals. It creates a full 'battle plan' *before* taking any action.

| Property | Value |
|---|---|
| Origin | Long lineage in classical AI (STRIPS, HTN); modern LangGraph idiom: [plan-and-execute tutorial](https://langchain-ai.github.io/langgraph/tutorials/plan-and-execute/plan-and-execute/) |

### High-level Workflow

1.  **Receive Goal:** The agent is given a complex task.
2.  **Plan:** A dedicated 'Planner' component analyzes the goal and generates an ordered list of sub-tasks required to achieve it. For example: `["Find fact A", "Find fact B"]`.
3.  **Execute:** An 'Executor' component takes the plan and carries out each sub-task in sequence, using tools as needed.
4.  **Synthesize:** Once all steps in the plan are complete, a final component synthesizes the results from the executed steps into a coherent final answer (including any calculation, such as "C from A and B").

### When to Use / Applications
*   **Multi-Step Workflows:** Ideal for tasks where the sequence of operations is known and critical, such as generating a report that requires fetching data, processing it, and then summarizing it.
*   **Project Management Assistants:** Decomposing a large goal like "launch a new feature" into sub-tasks for different teams.
*   **Educational Tutoring:** Creating a lesson plan to teach a student a specific concept, from foundational principles to advanced application.

### Strengths & Weaknesses
*   **Strengths:**
    *   **Structured & Traceable:** The entire workflow is laid out in advance, making the agent's process transparent and easy to debug.
    *   **Efficient:** Can be more efficient than ReAct for predictable tasks, as it avoids a reasoning call between every step.
*   **Weaknesses:**
    *   **Brittle to Change:** A pre-made plan can fail if the environment changes unexpectedly during execution. It's less adaptive than a ReAct agent, which can change its mind after every step.

> **Key Insight**: ReAct pays an LLM call per step to stay adaptive; a plan pays one call up front to stay cheap and inspectable. Neither is "better" - the task decides.

---
## 📦 Step 1: Environment Setup

Load the API keys, create the chat model through the course factory, and wrap Tavily in a `web_search` tool that both agents share.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: One model for the ReAct agent, planner, synthesizer and judge
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# SHARED TOOL: A Tavily web search both agents use
# ============================================================================
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(max_results=2)


@tool
def web_search(query: str) -> str:
    """Performs a web search using Tavily and returns the results as a string."""
    print(f"   🔎 TOOL: searching for '{query}'")
    return str(tavily_search_tool.invoke(query))


print(f"✅ Tool ready: {web_search.name}")

### 1.1 📏 Measuring What Each Agent Spends

To compare the agents fairly we count in code, not by eye. A LangChain **callback handler** sees every chat-model call made while a graph runs, so one small class can count LLM calls and the tokens they used. Tool calls are counted from each agent's final state.

In [ ]:
# ============================================================================
# CALL COUNTER: A callback that counts LLM calls and tokens during a run
# ============================================================================
import time

from langchain_core.callbacks import BaseCallbackHandler


class CallCounter(BaseCallbackHandler):
    """Counts chat-model calls and total tokens for everything run with it."""

    def __init__(self):
        self.llm_calls = 0
        self.tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.llm_calls += 1

    def on_llm_end(self, response, **kwargs):
        for generations in response.generations:
            for generation in generations:
                usage = getattr(generation.message, "usage_metadata", None) or {}
                self.tokens += usage.get("total_tokens", 0)


print("✅ CallCounter defined")

---
## 🔄 Part 2: The Baseline - A Reactive Agent (ReAct)

To appreciate the value of planning, we first need a baseline. We rebuild the ReAct agent from module 03 in a few lines. Its core feature is a loop where the agent's output is routed back to itself after every tool call, allowing it to reassess and decide its next move based on the latest information. The system prompt asks for **one tool call per turn**, and  enforces it, so every step costs a reasoning call - that is the behaviour we want to measure.

In [ ]:
# ============================================================================
# REACT AGENT: agent -> tools -> agent ... until the model stops calling tools
# ============================================================================
from typing import Annotated, TypedDict

from langchain_core.messages import SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.prebuilt import ToolNode, tools_condition


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


# parallel_tool_calls=False enforces one tool call per turn - a prompt alone does not:
# gpt-4o-mini happily batches several searches into one turn despite being told not to.
llm_with_tools = llm.bind_tools([web_search], parallel_tool_calls=False)

REACT_SYSTEM_PROMPT = SystemMessage(
    content=(
        "You are a helpful research assistant. You must call one and only one tool at a time. "
        "Do not call multiple tools in a single turn. After receiving the result from a tool, "
        "you will decide on the next step."
    )
)


def react_agent_node(state: AgentState):
    print("🧠 REACT AGENT: thinking...")
    response = llm_with_tools.invoke([REACT_SYSTEM_PROMPT] + state["messages"])
    return {"messages": [response]}


react_graph_builder = StateGraph(AgentState)
react_graph_builder.add_node("agent", react_agent_node)
react_graph_builder.add_node("tools", ToolNode([web_search]))
react_graph_builder.add_edge(START, "agent")
react_graph_builder.add_conditional_edges("agent", tools_condition)  # tool call -> "tools", else END
react_graph_builder.add_edge("tools", "agent")                        # the ReAct loop

react_agent_app = react_graph_builder.compile()
print("✅ Reactive (ReAct) agent compiled")

In [ ]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = react_agent_app.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

### 2.1 🧪 Testing the Reactive Agent on a Plan-Centric Problem

The task needs three data-gathering steps (one per capital), a fourth for the United States, then a calculation and a comparison. Every step is known in advance - exactly the kind of task a plan should handle well.

In [ ]:
# ============================================================================
# RUN THE REACT AGENT: Stream the steps, then measure the run
# ============================================================================
from IPython.display import Markdown, display

plan_centric_query = """
Find the population of the capital cities of France, Germany, and Italy.
Then calculate their combined total.
Finally, compare that combined total to the population of the United States, and say which is larger.
"""

react_counter = CallCounter()
start = time.perf_counter()
final_react_output = react_agent_app.invoke(
    {"messages": [("user", plan_centric_query)]},
    config={"callbacks": [react_counter], "recursion_limit": 20},
)
react_seconds = time.perf_counter() - start

react_tool_calls = sum(1 for m in final_react_output["messages"] if m.type == "tool")
print(f"\n📊 ReAct: {react_counter.llm_calls} LLM calls, {react_tool_calls} tool calls, "
      f"{react_counter.tokens} tokens, {react_seconds:.1f}s")
display(Markdown(final_react_output["messages"][-1].content))

Read the `🧠`/`🔎` lines above: the agent alternates **think → search → think → search**. Each search is followed by another LLM call that re-reads the whole conversation (including every earlier search result) to decide what to do next. For a task whose steps were predictable from the start, those in-between reasoning calls are overhead, and they grow the prompt each time.

Now read the **end** of the answer. In our runs (4 out of 4 in a row) the agent stopped after the three capitals with a sentence like *"Next, I will find the current population of the United States"* - written as text, not as a tool call. `tools_condition` ends the loop on any turn without a tool call, so the promised search never happens. A ReAct agent keeps its to-do list only in the model's head; a stronger prompt makes this rarer, but nothing in the graph checks that the task is finished. Part 5's judge will flag it.

---
## 📋 Part 3: The Advanced Approach - A Planning Agent

Now, let's build an agent that thinks before it acts. This agent has a dedicated **Planner** to create a complete task list, an **Executor** to carry out the plan, and a **Synthesizer** to assemble the final result.

### Key Concepts:
- **`Planner`**: one structured-output call that returns the list of search queries to run.
- **`Executor`**: runs the *next* query with the tool - no LLM call - and records the result.
- **`Synthesizer`**: one LLM call that turns all the collected results into the answer, including the arithmetic.

Because each plan step is a plain search query, the executor needs no LLM at all. Calculations and comparisons are left to the synthesizer, which sees every result at once.

In [ ]:
# ============================================================================
# PLAN SCHEMA + STATE: What the planner returns and what the graph carries
# ============================================================================
from typing import List, Optional

from pydantic import BaseModel, Field


class Plan(BaseModel):
    """A plan of web searches to run to answer the user's query."""

    steps: List[str] = Field(
        description="Ordered list of web search queries. Each query looks up exactly one fact."
    )


class PlanningState(TypedDict):
    user_request: str
    plan: Optional[List[str]]          # the queries still to run
    intermediate_steps: List[dict]     # {"query": ..., "result": ...} for each executed step
    final_answer: Optional[str]


print(f"✅ PlanningState fields: {list(PlanningState.__annotations__)}")

In [ ]:
# ============================================================================
# NODES: Planner (1 LLM call), Executor (no LLM), Synthesizer (1 LLM call)
# ============================================================================
planner_llm = llm.with_structured_output(Plan)


def planner_node(state: PlanningState):
    """Generates a plan of action to answer the user's request."""
    print("📋 PLANNER: decomposing task...")
    prompt = f"""You are an expert planner. Your job is to create a step-by-step plan to answer the user's request.
Each step in the plan must be a single web search query that looks up one fact.

**Instructions:**
1. Analyze the user's request.
2. List every fact that must be looked up - including any figure a comparison needs - one search query per fact.
3. Do not add steps for calculations or comparisons - those happen after all searches are done.

**Example:**
Request: "What is the capital of France and is its population bigger than Madrid's?"
Plan: ["capital of France", "population of Paris", "population of Madrid"]

**User's Request:**
{state['user_request']}
"""
    plan_result = planner_llm.invoke(prompt)
    print(f"📋 PLANNER: generated plan: {plan_result.steps}")
    return {"plan": plan_result.steps}


def executor_node(state: PlanningState):
    """Executes the next step in the plan - a tool call, no LLM involved."""
    query, remaining = state["plan"][0], state["plan"][1:]
    print(f"⚙️  EXECUTOR: running step '{query}'")
    result = web_search.invoke(query)
    return {
        "plan": remaining,  # pop the executed step
        "intermediate_steps": state["intermediate_steps"] + [{"query": query, "result": result}],
    }


def synthesizer_node(state: PlanningState):
    """Synthesizes the final answer from the intermediate steps."""
    print("🧩 SYNTHESIZER: generating final answer...")
    context = "\n".join(f"Search '{s['query']}' returned: {s['result']}" for s in state["intermediate_steps"])
    prompt = f"""You are an expert synthesizer. Based on the user's request and the collected data, provide a comprehensive final answer.
Do any calculation the request asks for, and show the numbers you used.

Request: {state['user_request']}
Collected Data:
{context}
"""
    return {"final_answer": llm.invoke(prompt).content}


print("✅ Planner, Executor, and Synthesizer nodes defined")

### 3.1 🏗️ Building the Planning Agent Graph

The flow is `plan → execute (looped) → synthesize`. A single router - a pure function of state - asks one question after planning and after every execution: *is the plan empty yet?*

In [ ]:
# ============================================================================
# GRAPH: plan -> execute (until the plan is empty) -> synthesize
# ============================================================================
def planning_router(state: PlanningState):
    return "execute" if state["plan"] else "synthesize"


planning_graph_builder = StateGraph(PlanningState)
planning_graph_builder.add_node("plan", planner_node)
planning_graph_builder.add_node("execute", executor_node)
planning_graph_builder.add_node("synthesize", synthesizer_node)

planning_graph_builder.add_edge(START, "plan")
planning_graph_builder.add_conditional_edges("plan", planning_router, {"execute": "execute", "synthesize": "synthesize"})
planning_graph_builder.add_conditional_edges("execute", planning_router, {"execute": "execute", "synthesize": "synthesize"})
planning_graph_builder.add_edge("synthesize", END)

planning_agent_app = planning_graph_builder.compile()
print("✅ Planning agent compiled")

In [ ]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = planning_agent_app.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

---
## ⚖️ Part 4: Head-to-Head Comparison

Run the planning agent on the same task, measured the same way.

In [ ]:
# ============================================================================
# RUN THE PLANNING AGENT: Same query, same measurements
# ============================================================================
plan_counter = CallCounter()
start = time.perf_counter()
final_planning_output = planning_agent_app.invoke(
    {"user_request": plan_centric_query, "intermediate_steps": []},
    config={"callbacks": [plan_counter]},
)
plan_seconds = time.perf_counter() - start

plan_tool_calls = len(final_planning_output["intermediate_steps"])
print(f"\n📊 Planning: {plan_counter.llm_calls} LLM calls, {plan_tool_calls} tool calls, "
      f"{plan_counter.tokens} tokens, {plan_seconds:.1f}s")
display(Markdown(final_planning_output["final_answer"]))

In [ ]:
# ============================================================================
# SIDE BY SIDE: The measured cost of each approach
# ============================================================================
print(f"{'':12}{'LLM calls':>10}{'tool calls':>12}{'tokens':>9}{'seconds':>9}")
print(f"{'ReAct':12}{react_counter.llm_calls:>10}{react_tool_calls:>12}{react_counter.tokens:>9}{react_seconds:>9.1f}")
print(f"{'Planning':12}{plan_counter.llm_calls:>10}{plan_tool_calls:>12}{plan_counter.tokens:>9}{plan_seconds:>9.1f}")

Read the table: the planning agent always makes exactly **two** LLM calls (planner + synthesizer), however many searches the plan contains. The ReAct agent makes one call per search **plus** one to answer, and each of those calls re-reads the growing conversation - compare the tokens column. Wall time is a different story: it is dominated by the searches, which the executor runs one after another, so it can go either way from run to run.

The plans also vary between runs: sometimes the planner looks up "capital of France" before "population of Paris", sometimes it goes straight to the city. Both are valid plans - and you can see which one you got before a single search ran.

The price of that efficiency is visible in the plan itself: it was fixed before the first search. If a search had returned the wrong thing, nothing in this graph would notice - that is what the next notebook fixes.

---
## 🧑‍⚖️ Part 5: Judging the Answers

Cost is only half the comparison - both agents must also get the task right. A first version of this judge asked one yes/no question per requirement ("does the answer state the U.S. population?"). With `gpt-4o-mini` it answered **yes** for an answer that only said *"Next, I will find the population of the United States"* - the question was in the task, so the judge "saw" it.

The fix is to ask the judge for **evidence, not verdicts**: copy each required figure out of the answer, or write `none`. Code then decides - a requirement passes only if its evidence exists, and the comparison is checked with real numbers.

In [ ]:
# ============================================================================
# ANSWER JUDGE: The LLM copies evidence, code computes the verdict
# ============================================================================
import re


class AnswerReview(BaseModel):
    """Evidence copied from the answer for each requirement of the task."""

    capital_figures: list[str] = Field(
        description="Each population figure the answer states for Paris, Berlin or Rome, copied exactly. Empty list if none."
    )
    combined_total: str = Field(
        description="The combined-total figure exactly as written in the answer, or 'none' if the answer does not state one."
    )
    us_population: str = Field(
        description="The United States population figure exactly as written in the answer, or 'none'. "
        "A sentence saying it WILL be looked up is not a figure."
    )
    conclusion: str = Field(description="The sentence in the answer that says which is larger, copied exactly, or 'none'.")


judge = llm.with_structured_output(AnswerReview)


def to_number(text: str) -> float | None:
    """'8,581,990' -> 8581990; '341.8 million' -> 341800000; 'none' -> None."""
    match = re.search(r"(\d[\d,]*\.?\d*)\s*(million|billion)?", text.replace("\u00a0", " "))
    if not match:
        return None
    value = float(match.group(1).replace(",", ""))
    return value * {"million": 1e6, "billion": 1e9}.get(match.group(2), 1)


def review_answer(answer: str) -> dict:
    ev = judge.invoke(f"Task:\n{plan_centric_query}\n\nAnswer to review:\n<answer>\n{answer}\n</answer>")
    total, us = to_number(ev.combined_total), to_number(ev.us_population)
    return {
        "three_capital_figures": len(ev.capital_figures) >= 3,
        "combined_total": total is not None,
        "us_population": us is not None,
        "correct_conclusion": ev.conclusion != "none" and total is not None and us is not None and us > total,
    }


for name, answer in [
    ("ReAct", final_react_output["messages"][-1].content),
    ("Planning", final_planning_output["final_answer"]),
]:
    checks = review_answer(answer)
    print(f"{'✅' if all(checks.values()) else '❌'} {name}: {checks}")

A ❌ names the requirement that answer missed - for the ReAct agent, typically the United States population it announced but never looked up. The planning agent listed that lookup in its plan before any search ran, so the synthesizer had the figure.

A judge that approves everything proves nothing. Show it an answer that is **incomplete** and check that it fails the right requirements:

In [ ]:
# ============================================================================
# JUDGE SANITY CHECK: An incomplete answer must fail
# ============================================================================
incomplete_answer = "Paris has about 2.1 million people and Berlin about 3.7 million."

checks = review_answer(incomplete_answer)
print(f"{'✅' if all(checks.values()) else '❌'} Incomplete answer -> failed: {[k for k, ok in checks.items() if not ok]}")

---
## 📝 Summary

In this notebook, we learned:

### 1. The planning pattern
- A planner decomposes the task into an explicit list of steps *before* any tool runs; an executor works through it; a synthesizer answers.
- It suits tasks whose steps are predictable; ReAct suits exploratory tasks where the next step depends on what was just found.

### 2. A ReAct baseline
- The `tools → agent` edge makes the agent reason again after every tool result - adaptive, but one LLM call per step.
- The loop ends on the first turn without a tool call, even when the model only *announced* its next step - nothing tracks what is left to do.

### 3. A plan-and-execute agent
- Plan steps that are plain search queries let the executor run without an LLM; calculations belong to the synthesizer.
- A single pure router (`is the plan empty?`) drives the whole loop.

### 4. A measured comparison
- A callback handler counts LLM calls and tokens for any graph run; tool calls come from the final state.
- The planning agent's LLM calls stay at two regardless of plan length; tokens and wall time depend on how many searches the plan contains.

### 5. Judging the answers
- Ask the judge to copy evidence (or `none`), then compute the verdict in code - a yes/no question can be answered from the task instead of the answer.
- A deliberately incomplete answer proves the judge can fail.

### Next Steps
- **`02_planning_decompose_execute_replan.ipynb`**: evaluate every step's result and rewrite the rest of the plan when a step fails